# Solar Filament Segmentation: Pipeline Walkthrough

End-to-end walkthrough of this project for the [Solar Filament Segmentation Challenge 2026](https://www.kaggle.com/competitions/filament-segmentation-2026): data exploration, splits, training, evaluation, and submission generation.

This is a narrated summary of the whole project, not just the final model — including the dead ends and negative results, since those were as much a part of the real process as the wins, and understanding *why* something didn't work shaped what we tried next. Every run mentioned here has its own `README.md` (`V1/` through `V6/`, `EdgeAttNet/`) with the full detail.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "../scripts")
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from pycocotools import mask as mask_util
from pycocotools.coco import COCO

from dataset import load_image_tensor, load_image_uint8, read_split
from fast_pq import build_ground_truth, pq_from_stats, sweep_stats
from metrics import get_overlap_df, pq_breakdown
from model import get_device
from predict import instances_to_df, load_model, merge_nearby_instances, predict_instances

DATA_ROOT = Path("..")
device = get_device()
print(f"device: {device}")

## 2. Data exploration

In [ ]:
ann_path = DATA_ROOT / "data/raw/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
coco = COCO(str(ann_path))
d = json.load(open(ann_path))

n_images = len(set(im["file_name"] for im in d["images"]))
n_entries = len(d["images"])  # one entry per (annotator, image) pair -- some images have several annotators
print(f"{n_images} distinct images, {n_entries} annotator-image entries, {len(d['annotations'])} filament annotations")

years = Counter(im["file_name"][:4] for im in d["images"])
station_names = {"B": "Big Bear", "M": "Mauna Loa", "L": "Learmonth", "U": "Udaipur", "T": "El Teide", "C": "Cerro Tololo"}
stations = Counter(station_names.get(im["file_name"][-7], "?") for im in d["images"])
print("\nby year:", dict(sorted(years.items())))
print("by station:", dict(sorted(stations.items())))
# Note (see EdgeAttNet/README.md investigation): images are heavily concentrated in 2011-2017 (~88%),
# with very little from 2018 onward -- a real, if hard-to-quantify with this few late-year examples,
# possible distribution gap between our training data and however recent the hidden test set is.

In [ ]:
# a sample image with its ground-truth filament outlines
sample_id = coco.getImgIds()[0]
info = coco.imgs[sample_id]
image = load_image_uint8(DATA_ROOT / "data/raw/train/train_images", info["file_name"]).squeeze(0).numpy()
anns = coco.loadAnns(coco.getAnnIds(imgIds=[sample_id]))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image, cmap="gray")
for a in anns:
    ax.contour(coco.annToMask(a), colors="red", linewidths=0.8)
ax.set_title(f"{info['file_name']}  ({len(anns)} annotated filaments)")
ax.axis("off")
plt.show()

## 3. Train/val split

In [ ]:
train_ids = read_split(DATA_ROOT / "data/splits/train.txt")
val_ids = read_split(DATA_ROOT / "data/splits/val.txt")
train_files = {coco.imgs[i]["file_name"] for i in train_ids}
val_files = {coco.imgs[i]["file_name"] for i in val_ids}
print(f"train: {len(train_ids)} annotator-entries over {len(train_files)} distinct images")
print(f"val:   {len(val_ids)} annotator-entries over {len(val_files)} distinct images")

assert not (train_files & val_files), "train and val must not share any images"
print("confirmed: zero image overlap between the train and val splits (split by image, not by annotator-entry,"
      " so a multi-annotator image can't leak across the split)")

## 4. Training

This project ran 6 numbered Mask R-CNN training runs plus two side investigations into alternative approaches. Full detail for each lives in its own `README.md`; this is the summary.

**V1 (baseline):** fine-tuned Mask R-CNN (ResNet-50 FPN, COCO-pretrained). First working pipeline. PQ ~38.6%.

**V2:** added augmentation, one-annotator-per-epoch sampling, and dropped filaments only one annotator drew (an agreement filter). PQ 39.8%.

**V3:** widened the IoU margin used to decide ambiguous box/mask matches during training, hoping to help borderline cases. This was a regression (39.2%, worse than V2) — reverted, but kept and documented as a negative result rather than deleted.

**V4:** V2's recipe consolidated and trained for 30 epochs instead of 40 (V2 peaked around epoch 18-20; the extra epochs were only overfitting). PQ 40.3%.

**V5:** added a recall-biased Tversky mask loss, motivated by a direct measurement of what V4 was actually getting wrong (of its missed filaments, ~59% were detected with an imprecise mask, not a totally missing one — a mask-shape problem, not a detection problem). PQ 40.8%, our best trained model. Test-time augmentation (8-view pooling) was also tried on top of V5 and added nothing further — TTA and the Tversky loss turned out to be rescuing the same underlying failure mode, not different ones.

**V6a / V6b:** two more single-variable tests on top of V5, both motivated by a direct measurement that filaments V5 misses *entirely* (zero overlap with any prediction) are ~44% smaller in area and ~23% fainter than the ones it catches — shape/elongation show no difference at all, and image blur/weather was checked and ruled out separately (correlation ~0 with miss rate). V6a reweighted the mask loss per-instance to upweight small filaments; V6b excluded 3 training images that an independent, human-labeled quality dataset flagged as having real visible defects. **Neither beat V5** (40.6% and 39.9%) — both showed a promising early lead over V5 that did not hold up by the final epoch, a useful reminder not to call a winner early.

**Side investigation — EdgeAttNet:** a published, purpose-built filament-segmentation architecture (U-Net + edge-guided attention), reimplemented from its paper's architecture description — deliberately *not* using the authors' released weights, since those were trained on a MAGFiLO split that very likely overlaps this competition's hidden test set. Trained fresh, only on our own permitted data: 34.2% PQ, well below V5. Diagnosis (SQ/RQ decomposition): its mask *shape* quality was equal to V5's (66.5% vs 66.6%), but its *recognition* quality was much worse (47% vs 62%) — it is a semantic-segmentation model with no instance-level training signal, and produces 2.4x more false-positive detections than V5 for a similar number of real matches. Ensembling it with V5 (both naive score-pooling and an asymmetric "V5 first, EdgeAttNet fills gaps only" version, swept over multiple thresholds) never beat V5 alone in any tested configuration.

**The one real win — post-processing, not training.** Running the same SQ/RQ diagnosis on V5's *own* errors found that 78% of its false positives are near-misses of real filaments: Mask R-CNN sometimes emits multiple overlapping/adjacent detections for what should be one filament. Merging nearby detections after score-thresholding (`predict.merge_nearby_instances`, dilate-to-decide-grouping, keep original pixels) gives **+0.8pp PQ for free** — verified against the official scorer, stable across a wide range of the one tunable radius parameter. This is what the final submission uses.

In [ ]:
import csv

runs = [
    ("V2", "V2/checkpoints/run1/history.csv"),
    ("V3 (regression, reverted)", "V3/checkpoints/run1/history.csv"),
    ("V4", "V4/checkpoints/run1/history.csv"),
    ("V5 (best trained model)", "V5/checkpoints/run1/history.csv"),
    ("V6a, size-weighted loss", "V6/checkpoints/run_a_sizeweighted/history.csv"),
    ("V6b, anomaly-excluded", "V6/checkpoints/run_b_no_anomalous/history.csv"),
    ("EdgeAttNet (alt. architecture)", "EdgeAttNet/checkpoints/run1/history.csv"),
]
print(f"{'run':32s} {'best epoch':>10s} {'best PQ':>8s}")
for name, path in runs:
    full_path = DATA_ROOT / path
    if not full_path.exists():
        print(f"{name:32s} (checkpoint not present locally)")
        continue
    rows = [r for r in csv.DictReader(open(full_path)) if r.get("pq")]
    best = max(rows, key=lambda r: float(r["pq"]))
    print(f"{name:32s} {best['epoch']:>10s} {100*float(best['pq']):>7.1f}%")
print(f"\n{'V5 + merge_radius=2 (final)':32s} {'--':>10s} {'41.7%':>8s}   <- see section 5, verified against the official scorer")

## 5. Evaluation

The official scoring code (`scripts/metrics.py`) is copied verbatim from the organizers' own scoring notebook. A faster approximate scorer (`fast_pq.py`) is used for fine cutoff/parameter sweeps during development, but every result that actually gets used is re-verified against the official code before being trusted — that discipline caught real bugs earlier in this project (see `V5/README.md`'s note on a TTA overlap bug).

This cell reproduces our final configuration end-to-end on the validation set: V5's trained model, at its established best score cutoff, plus the merge-radius post-processing described above. It takes a few minutes (full inference over the validation images).

In [ ]:
cfg = yaml.safe_load(open(DATA_ROOT / "V5/configs/maskrcnn_v5.yaml"))
d_cfg = cfg["data"]
model = load_model(cfg, str(DATA_ROOT / "V5/checkpoints/run1/best_pq.pt"), device)

val_ids_eval = read_split(DATA_ROOT / d_cfg["val_split"])
coco_eval = COCO(str(DATA_ROOT / d_cfg["annotations"]))
files = sorted({coco_eval.imgs[i]["file_name"] for i in val_ids_eval})
gt = build_ground_truth(coco_eval, val_ids_eval)

V5_CUTOFF, MERGE_RADIUS = 0.825, 2
instances = {}
for f in files:
    image = load_image_tensor(DATA_ROOT / d_cfg["train_images"], f)
    preds = predict_instances(model, image, device, mask_threshold=cfg["predict"]["mask_threshold"])
    kept = [(s, r) for s, r in preds if s >= V5_CUTOFF]
    instances[f] = merge_nearby_instances(kept, image.shape[1:], MERGE_RADIUS)

stats = sweep_stats(instances, gt, [-1.0])   # already thresholded+merged -- keep everything with this sentinel
sum_iou, matched, fp, fn = stats[0]
pq = pq_from_stats(stats)[0]
sq = sum_iou / matched
rq = matched / (matched + 0.5 * fp + 0.5 * fn)
print(f"fast_pq:  PQ={100*pq:.2f}%  SQ={100*sq:.1f}%  RQ={100*rq:.1f}%  matched={int(matched)} FP={int(fp)} FN={int(fn)}")

# official cross-check -- the ground-truth id format ("<entry_id>_<k>") must use the raw COCO image id, which is
# itself a composite "<annotator>-<image>" string in this dataset -- NOT derived from the file name (a mistake
# almost shipped once already, see the ensemble-evaluation work earlier in this project)
gt_df = pd.DataFrame([(f"{e}_{k}", r) for fname, per in gt.items() for e, rl in per.items()
                      for k, r in enumerate(rl)], columns=["filament_id", "segmentation_rle"])
pred_df = instances_to_df({Path(fname).stem: v for fname, v in instances.items()}, -1.0)
official = pq_breakdown(get_overlap_df(gt_df, pred_df))["pq"]
print(f"official: PQ={100*official:.2f}%   {'MATCH' if abs(pq - official) < 1e-3 else 'MISMATCH -- investigate'}")

## 6. Submission

df = pd.read_csv(DATA_ROOT / "submissions/v5_run1_merge2_cutoff0.825.csv")
print(f"{len(df)} predicted filaments")

df["image"] = df["filament_id"].str.rsplit("_", n=1).str[0]
print(f"{df['image'].nunique()} distinct test images with at least one prediction (out of 180 total)")

overlap_found = False
for image, group in df.groupby("image"):
    taken = np.zeros((2048, 2048), dtype=bool)
    for rle in group["segmentation_rle"]:
        m = mask_util.decode({"size": [2048, 2048], "counts": rle.encode("ascii")}).astype(bool)
        if (m & taken).any():
            overlap_found = True
        taken |= m

print("RESULT:", "OVERLAP FOUND -- DO NOT SUBMIT" if overlap_found else "no overlaps found in any image -- safe to submit")